# w10b — Comparing Two Means in Python
**ENGR 1451/2451 · Exploratory Data Science · Fall 2026**

This short notebook follows the w10b slides and emphasizes choosing a method from the **study design**:

1. **Heat treatment:** compare independent groups using sample sizes, means, and sample SDs.
2. **Vitamin C versus placebo:** analyze a randomized experiment using textbook summary statistics.
3. **Study app:** read individual observations from `study_app_scores.csv`.
4. **Paired measurements:** compare two measurements made on the same units.

For independent groups with unknown population SDs, we use **Welch's two-sample $t$-test** (`equal_var=False`). Choose the group order and the direction of the alternative **before** examining the result.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.precision", 4)
alpha = 0.05

## 1. Heat treatment — summary statistics

A laboratory randomly assigns 24 steel coupons to a **new heat treatment** or a **standard-cycle control**.

| Group | $n$ | Mean hardness (HV) | Sample SD (HV) |
|---|---:|---:|---:|
| New heat treatment | 12 | 330 | 18 |
| Standard cycle | 12 | 314 | 14 |

Let $\Delta=\mu_T-\mu_C$. Test

$$H_0:\Delta=0 \qquad \text{against} \qquad H_A:\Delta>0.$$

For independent groups,

$$\widehat{\mathrm{SE}}(\bar X_T-\bar X_C)
=\sqrt{\frac{s_T^2}{n_T}+\frac{s_C^2}{n_C}}.$$

`stats.ttest_ind_from_stats()` uses the reported sample summaries. The SD inputs must be **sample SDs** computed with divisor $n-1$.

In [ ]:
# Treatment is group 1 and control is group 2.
n_t, mean_t, sd_t = 12, 330.0, 18.0
n_c, mean_c, sd_c = 12, 314.0, 14.0

difference = mean_t - mean_c
se_difference = np.sqrt(sd_t**2 / n_t + sd_c**2 / n_c)
t_manual = difference / se_difference

test_heat = stats.ttest_ind_from_stats(
    mean1=mean_t, std1=sd_t, nobs1=n_t,
    mean2=mean_c, std2=sd_c, nobs2=n_c,
    equal_var=False,
    alternative="greater",
)

print(f"Observed difference: {difference:.1f} HV")
print(f"Estimated SE:        {se_difference:.3f} HV")
print(f"Manual t:            {t_manual:.3f}")
print(f"SciPy t:             {test_heat.statistic:.3f}")
print(f"One-sided P-value:   {test_heat.pvalue:.4f}")
print("Decision:", "reject H0" if test_heat.pvalue < alpha else "fail to reject H0")

### Two-sided confidence interval for the treatment effect

The slides also report a two-sided 95% confidence interval for $\Delta=\mu_T-\mu_C$.

`ttest_ind_from_stats()` returns the test statistic and P-value, but not a confidence interval. With summary statistics, we therefore calculate Welch's approximate degrees of freedom and pass the estimated difference and SE to `stats.t.interval()`.

You should understand **why degrees of freedom are needed**, but you do not need to memorize the Welch formula.

In [ ]:
def welch_df(sd1, n1, sd2, n2):
    """Welch-Satterthwaite approximate degrees of freedom."""
    v1 = sd1**2 / n1
    v2 = sd2**2 / n2
    return (v1 + v2)**2 / (v1**2 / (n1 - 1) + v2**2 / (n2 - 1))

df_welch = welch_df(sd_t, n_t, sd_c, n_c)
ci_heat = stats.t.interval(
    0.95,
    df=df_welch,
    loc=difference,
    scale=se_difference,
)

two_sided_heat = stats.ttest_ind_from_stats(
    mean1=mean_t, std1=sd_t, nobs1=n_t,
    mean2=mean_c, std2=sd_c, nobs2=n_c,
    equal_var=False,
    alternative="two-sided",
)

print(f"Welch df:             {df_welch:.2f}")
print(f"Two-sided 95% CI:     [{ci_heat[0]:.2f}, {ci_heat[1]:.2f}] HV")
print(f"Two-sided P-value:    {two_sided_heat.pvalue:.4f}")

## 2. Vitamin C versus placebo — randomized experiment

In the w10b slides, **200 subjects are randomly assigned** to vitamin C or placebo. The response is the **number of colds per subject** (not how long a cold lasts).

| Group | $n$ | Average number of colds | SD (book) |
|---|---:|---:|---:|
| Vitamin C | 100 | 2.3 | 3.1 |
| Placebo | 100 | 2.6 | 2.9 |

Let $\Delta=\mu_{\mathrm{vitamin\ C}}-\mu_{\mathrm{placebo}}$. We test

$$H_0:\Delta=0 \qquad \text{against} \qquad H_A:\Delta<0.$$

Since **fewer** colds would support a benefit, use `alternative="less"`. We only have group summaries, so use `stats.ttest_ind_from_stats()`.

In [ ]:
# Vitamin C is group 1; placebo is group 2.
n_vitc, mean_vitc, sd_vitc = 100, 2.3, 3.1
n_placebo, mean_placebo, sd_placebo = 100, 2.6, 2.9

vitamin_c_test = stats.ttest_ind_from_stats(
    mean1=mean_vitc, std1=sd_vitc, nobs1=n_vitc,
    mean2=mean_placebo, std2=sd_placebo, nobs2=n_placebo,
    equal_var=False,
    alternative="less",
)

print(f"Observed difference: {mean_vitc - mean_placebo:.1f} colds per person")
print(f"Welch t:             {vitamin_c_test.statistic:.3f}")
print(f"One-sided P-value:   {vitamin_c_test.pvalue:.3f}")
print("Decision:", "reject H0" if vitamin_c_test.pvalue < alpha else "fail to reject H0")

**Interpretation check**

- **Why random assignment?** It helps make the treatment and control groups comparable, reducing systematic differences and supporting a causal interpretation of the comparison.
- **Does $P\approx0.24$ prove that vitamin C has no effect?** No. Since $P>0.05$, we fail to reject $H_0$. This experiment does not provide convincing statistical evidence that vitamin C reduces the **average number of colds**, but a nonsignificant result does **not** prove the effect is zero.

**Remember:** A well-designed randomized experiment can support causal reasoning even if the observed difference is not statistically significant.

## 3. Study app — raw observations from a CSV file

Fifty students are randomly assigned to a **study app** or a **control** condition. Each row of `study_app_scores.csv` contains one student's group and score.

Let $\Delta=\mu_{\mathrm{app}}-\mu_{\mathrm{control}}$ and test

$$H_0:\Delta=0 \qquad \text{against} \qquad H_A:\Delta>0.$$

We first verify the group summaries and then use `stats.ttest_ind()` on the individual observations. In pandas, `.std()` uses the sample-SD convention ($n-1$).

In [ ]:
csv_path = Path("data/study_app_scores.csv")
df = pd.read_csv(csv_path)
summary = df.groupby("group")["score"].agg(n="count", mean="mean", sd="std")
display(df.head())
display(summary)

# The order is app minus control because the alternative is "greater".
app = df.loc[df["group"] == "app", "score"]
control = df.loc[df["group"] == "control", "score"]

one_sided_app = stats.ttest_ind(
    app, control,
    equal_var=False,
    alternative="greater",
)

print(f"Observed difference: {app.mean() - control.mean():.1f} points")
print(f"Welch t:             {one_sided_app.statistic:.3f}")
print(f"Welch df:            {one_sided_app.df:.2f}")
print(f"One-sided P-value:   {one_sided_app.pvalue:.4f}")
print("Decision:", "reject H0" if one_sided_app.pvalue < alpha else "fail to reject H0")

**Interpretation:** The app group scores 5 points higher on average, but the one-sided $P\approx0.0656$ exceeds 0.05. We **fail to reject** $H_0$. This is insufficient evidence of a higher population mean; it does not prove that the app has no benefit.

Because students were randomly assigned, the design can support a causal comparison. Statistical evidence and study design answer different questions.

## 4. Paired data — compare within each unit

For before/after or matched observations, do **not** treat the two columns as independent samples. Define a difference for each unit,

$$d_i=\text{after}_i-\text{before}_i,$$

and test the mean of those differences. `stats.ttest_rel(after, before)` performs this paired $t$-test. The arrays must be aligned so that the two values at each position belong to the same unit.

In [ ]:
# Hypothetical hardness measurements on the same 10 coupons.
before = np.array([302, 315, 310, 298, 320, 305, 312, 307, 299, 316])
after  = np.array([308, 313, 315, 299, 328, 305, 315, 311, 298, 323])

differences = after - before
paired_test = stats.ttest_rel(after, before, alternative="greater")

print("Within-coupon differences:", differences)
print(f"Mean difference: {differences.mean():.2f} HV")
print(f"SD of differences: {differences.std(ddof=1):.2f} HV")
print(f"Paired t = {paired_test.statistic:.3f}, df = {paired_test.df}")
print(f"One-sided P-value = {paired_test.pvalue:.4f}")

### Python function reference

| Function or method | When to use it |
|---|---|
| `stats.ttest_ind_from_stats(...)` | Independent groups when only $n$, sample mean, and sample SD are available. |
| `stats.ttest_ind(a, b, equal_var=False, ...)` | Individual measurements from two independent groups (Welch's test). |
| `stats.ttest_rel(after, before, ...)` | Paired or matched measurements aligned by unit. |
| `stats.ttest_1samp(x, popmean=mu0, ...)` | One sample compared with a fixed population mean. |
| `result.confidence_interval(confidence_level=0.95)` | Confidence interval from a raw-data SciPy $t$-test result. |
| `stats.t.interval(0.95, df=..., loc=..., scale=...)` | Confidence interval when the estimate, SE, and degrees of freedom are already known. |
| `stats.norm.cdf(z)` / `stats.norm.sf(z)` | Lower / upper tail for a $z$ statistic calculated separately. |

For the SciPy $t$-test functions, `alternative` can be:

- `"two-sided"`: the population means differ (default),
- `"greater"`: the first input has the larger population mean,
- `"less"`: the first input has the smaller population mean.

## Your turn — battery cooling method

Two independent sets of battery modules are tested under the same load. Lower peak temperature is better.

| Cooling method | $n$ | Mean peak temperature ($^\circ$C) | Sample SD ($^\circ$C) |
|---|---:|---:|---:|
| New method | 10 | 61.2 | 3.4 |
| Standard method | 12 | 64.5 | 4.1 |

Let $\Delta=\mu_{\mathrm{new}}-\mu_{\mathrm{standard}}$.

1. State $H_0$ and $H_A$ for the claim that the new method lowers mean peak temperature.
2. Use `stats.ttest_ind_from_stats(..., equal_var=False, alternative="less")`.
3. Report the observed difference, $t$ statistic, P-value, and decision at $\alpha=0.05$.
4. Explain why this is an independent two-sample test rather than a paired test.

In [ ]:
n_new, mean_new, sd_new = 10, 61.2, 3.4
n_standard, mean_standard, sd_standard = 12, 64.5, 4.1

# 1. State H0 and HA in a markdown cell or comment.

# 2. Run Welch's independent two-sample t-test.
# YOUR CODE HERE

# 3. Report the difference, t statistic, P-value, and decision.
# YOUR CODE HERE

# 4. Why are these observations independent rather than paired?
# YOUR ANSWER: